In [2]:
# TEP pressure-transition study: complete human-authored scientific input
#
# This notebook is the authoritative scientific specification. Execution may
# implement mechanics needed to realize these rules, but it must not introduce
# an absent scientific rule or silently resolve an open scientific choice.

import featuregraph as fg
import pandas as pd

# ---------------------------------------------------------------------------
# Study scope
# ---------------------------------------------------------------------------

study_scope = {
    "dataset": "Tennessee Eastman Process (featuregraph.datasets.eastman)",
    "fault_number": 2,
    "simulation_run": 10,
    "signal": "reactor_pressure",
    "time_column": "time_(h)",
    "purpose": (
        "Construct inspectable candidate reactor-pressure transition intervals "
        "from one selected simulated run."
    ),
    "unit_of_analysis": "one simulated process run",
}

# ---------------------------------------------------------------------------
# Human-selected construction parameters
# ---------------------------------------------------------------------------

construction_parameters = {
    "smooth_window_minutes": 20,
    "rate_eps_pressure_units_per_hour": 0.75,
    "rolling_centered": True,
    "rolling_min_periods": "full window",
}

# ---------------------------------------------------------------------------
# Observations and preprocessing
# ---------------------------------------------------------------------------

observation_definition = {
    "sample_index": "source dataframe index",
    "time_hours": "time_(h)",
    "reactor_pressure_raw": "reactor_pressure",
    "sample_interval_minutes": (
        "median positive difference of time_hours, multiplied by 60"
    ),
    "smooth_window_samples": (
        "round(smooth_window_minutes / sample_interval_minutes)"
    ),
    "reactor_pressure_smooth": (
        "centered rolling arithmetic mean using smooth_window_samples and "
        "requiring the full window"
    ),
    "reactor_pressure_change": (
        "first difference of reactor_pressure_smooth"
    ),
    "reactor_pressure_rate": (
        "reactor_pressure_change divided by the first difference of time_hours"
    ),
    "reactor_pressure_valid": (
        "smooth pressure and pressure rate are both non-missing"
    ),
}

# ---------------------------------------------------------------------------
# Primitive mutually exclusive directional states
# ---------------------------------------------------------------------------

state_definitions = {
    "reactor_pressure_rising": (
        "valid and reactor_pressure_rate > +0.75 pressure units/hour"
    ),
    "reactor_pressure_falling": (
        "valid and reactor_pressure_rate < -0.75 pressure units/hour"
    ),
    "reactor_pressure_inactive": (
        "valid and abs(reactor_pressure_rate) <= 0.75 pressure units/hour"
    ),
}

# ---------------------------------------------------------------------------
# Boundary events and candidate identity
# ---------------------------------------------------------------------------

event_definitions = {
    "enter_reactor_pressure_rising": (
        "integer first difference of reactor_pressure_rising equals +1"
    ),
    "exit_reactor_pressure_rising": (
        "integer first difference of reactor_pressure_rising equals -1"
    ),
}

candidate_definition = {
    "identity_column": "exit_reactor_pressure_rising_id",
    "identity_rule": "cumulative sum of exit_reactor_pressure_rising",
    "membership_rule": (
        "samples sharing an exit_reactor_pressure_rising_id form one candidate "
        "interval; boundary fragments remain visible"
    ),
    "interpretation": (
        "candidate interval for scientific inspection, not yet a validated "
        "complete pressure-transition episode"
    ),
}

candidate_properties = {
    "start_index": "minimum sample_index",
    "end_index": "maximum sample_index",
    "start_time_hours": "minimum time_hours",
    "end_time_hours": "maximum time_hours",
    "duration_hours": "end_time_hours - start_time_hours",
    "reactor_pressure_rising_samples": "sum of rising state",
    "reactor_pressure_falling_samples": "sum of falling state",
    "reactor_pressure_inactive_samples": "sum of inactive state",
    "enter_rising_count": "sum of enter-rising events",
    "exit_rising_count": "sum of exit-rising events",
}

# ---------------------------------------------------------------------------
# Validation, outputs, and claim limits
# ---------------------------------------------------------------------------

validation_requirements = [
    "time_hours must be strictly increasing",
    "sample interval must be finite and positive",
    "smooth_window_samples must be at least one",
    "raw reactor pressure must not be modified",
    "exactly one directional state must hold at every valid sample",
    "no directional state may hold at an invalid sample",
    "enter and exit event columns must match the defined boolean transitions",
    "candidate aggregation must account for every source row exactly once",
    "boundary fragments must be retained and identified rather than discarded",
]

requested_outputs = [
    "observation-level table with preprocessing, states, events, and candidate ID",
    "candidate-level summary table using the specified properties",
    "validation report",
    "fragmentation diagnostics, including candidate count and boundary fragments",
]

supported_claims = [
    "the rules reproducibly construct candidate pressure intervals for the selected run",
    "each candidate can be traced to its supporting observations and construction parameters",
    "fragmentation under the selected threshold can be measured and reported",
]

unsupported_claims = [
    "a candidate interval is automatically a complete physical episode",
    "the construction detects or diagnoses Fault 2",
    "the selected threshold or smoothing window is optimal",
    "the rules generalize to other runs, faults, signals, plants, or real operations",
    "causal, prognostic, or safety conclusions",
]

unresolved_scientific_choices = [
    "minimum state persistence",
    "whether nearby directional runs should be merged",
    "criteria for a complete pressure-transition episode",
    "criteria for excluding or classifying boundary fragments",
    "cross-run and cross-fault validation design",
]

execution_contract = {
    "allowed": [
        "implement the stated transformations and aggregations",
        "add mechanical assertions and provenance fields",
        "report fragmentation and boundary status",
        "format the requested tables and diagnostics",
    ],
    "must_ask_before": [
        "adding persistence, debouncing, hysteresis, or merge rules",
        "changing the 20-minute centered mean",
        "changing the 0.75 pressure-units/hour threshold",
        "discarding candidates or boundary fragments",
        "promoting candidates to validated complete episodes",
        "making claims beyond supported_claims",
    ],
    "preserve": [
        "all selected parameters, definitions, open choices, and claim limits",
        "traceability from candidate rows to source observations",
    ],
}
